In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🧠 OmniBrain — Database & Qdrant Verification Notebook\n",
    "\n",
    "This interactive notebook validates:\n",
    "1. **Async SQLite / PostgreSQL Connection** & ORM table creation.\n",
    "2. **Atomic CRUD Operations** (Users, Documents, Chat Sessions, Messages).\n",
    "3. **Qdrant Vector Store Operations** (Collection Initialization, Batch Upsert, Cosine Similarity, and Hybrid Search)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import asyncio\n",
    "import uuid\n",
    "import numpy as np\n",
    "from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession\n",
    "\n",
    "# Import OmniBrain Database modules\n",
    "try:\n",
    "    from Database.database import Base\n",
    "    from Database.models import User, Document, ChatSession, ChatMessage, AuditLog\n",
    "    from Database.schemas import UserCreate, UserRole, ChatSessionCreate\n",
    "    from Database.vector_store import QdrantVectorStore\n",
    "    import Database.crud as crud\n",
    "except ImportError:\n",
    "    from database import Base\n",
    "    from models import User, Document, ChatSession, ChatMessage, AuditLog\n",
    "    from schemas import UserCreate, UserRole, ChatSessionCreate\n",
    "    from vector_store import QdrantVectorStore\n",
    "    import crud\n",
    "\n",
    "print(\"✅ All OmniBrain modules successfully imported!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Test Relational Database & Table Creation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Set up an in-memory SQLite database for testing\n",
    "test_engine = create_async_engine(\"sqlite+aiosqlite:///:memory:\", echo=False)\n",
    "SessionLocal = async_sessionmaker(test_engine, expire_on_commit=False, class_=AsyncSession)\n",
    "\n",
    "async def initialize_tables():\n",
    "    async with test_engine.begin() as conn:\n",
    "        await conn.run_sync(Base.metadata.create_all)\n",
    "    print(\"✅ Relational database tables (users, documents, chat_sessions, chat_messages, audit_logs) created!\")\n",
    "\n",
    "await initialize_tables()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Test User & Document CRUD Operations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "async def test_user_and_doc_flow():\n",
    "    async with SessionLocal() as db:\n",
    "        # 1. Create a user\n",
    "        user_in = UserCreate(\n",
    "            email=\"engineer@omnibrain.ai\",\n",
    "            username=\"omnibrain_dev\",\n",
    "            full_name=\"OmniBrain Engineer\",\n",
    "            password=\"supersecret123\",\n",
    "            role=UserRole.ADMIN\n",
    "        )\n",
    "        user = await crud.create_user(db, user_in, hashed_password=\"$2b$12$hashedpassworddemo\")\n",
    "        print(f\"👤 Created User: {user.username} (ID: {user.id})\")\n",
    "\n",
    "        # 2. Ingest document metadata\n",
    "        doc = await crud.create_document(\n",
    "            db=db,\n",
    "            user_id=user.id,\n",
    "            title=\"OmniBrain_System_Whitepaper.pdf\",\n",
    "            file_type=\"application/pdf\",\n",
    "            file_size_bytes=5242880,\n",
    "            page_count=28,\n",
    "            tags=[\"architecture\", \"rag\", \"multimodal\"],\n",
    "            meta_info={\"author\": \"AI Research Lab\"}\n",
    "        )\n",
    "        print(f\"📄 Ingested Document: {doc.title} (Status: {doc.status})\")\n",
    "\n",
    "        # 3. Update document status\n",
    "        updated_doc = await crud.update_document_status(db, doc.id, status=\"indexed\", chunk_count=120)\n",
    "        print(f\"🔄 Updated Status: {updated_doc.status}, Total Chunks: {updated_doc.chunk_count}\")\n",
    "\n",
    "await test_user_and_doc_flow()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Test Multi-Turn Chat Sessions & Citation Tracking"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "async def test_chat_session_flow():\n",
    "    async with SessionLocal() as db:\n",
    "        user = await crud.get_user_by_username(db, \"omnibrain_dev\")\n",
    "        \n",
    "        # 1. Create a Chat Session\n",
    "        session_in = ChatSessionCreate(title=\"OmniBrain Architecture Q&A\")\n",
    "        session = await crud.create_chat_session(db, user.id, session_in)\n",
    "        print(f\"💬 Created Chat Session: '{session.title}' (ID: {session.id})\")\n",
    "\n",
    "        # 2. Append User Query\n",
    "        msg1 = await crud.create_chat_message(\n",
    "            db=db,\n",
    "            session_id=session.id,\n",
    "            role=\"user\",\n",
    "            content=\"How is hybrid vector search ranked in OmniBrain?\"\n",
    "        )\n",
    "\n",
    "        # 3. Append Assistant Response with Citations\n",
    "        citations = [\n",
    "            {\n",
    "                \"source\": \"OmniBrain_System_Whitepaper.pdf\",\n",
    "                \"page\": 14,\n",
    "                \"snippet\": \"Hybrid ranking calculates 0.7 * text_score + 0.3 * image_score.\",\n",
    "                \"score\": 0.94\n",
    "            }\n",
    "        ]\n",
    "        msg2 = await crud.create_chat_message(\n",
    "            db=db,\n",
    "            session_id=session.id,\n",
    "            role=\"assistant\",\n",
    "            content=\"OmniBrain fuses cosine similarity scores across modalities.\",\n",
    "            citations=citations,\n",
    "            tokens_used=64\n",
    "        )\n",
    "\n",
    "        # Retrieve and display history\n",
    "        messages = await crud.get_messages_by_session(db, session.id)\n",
    "        print(f\"📜 Retrieved {len(messages)} messages from session history.\")\n",
    "        for m in messages:\n",
    "            print(f\"  [{m.role.upper()}]: {m.content}\")\n",
    "\n",
    "await test_chat_session_flow()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Test Qdrant Vector Store Simulation (Upsert & Hybrid Search)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate mock 768-dim text and 512-dim image embeddings\n",
    "np.random.seed(42)\n",
    "sample_text_vector = np.random.rand(768).tolist()\n",
    "sample_image_vector = np.random.rand(512).tolist()\n",
    "\n",
    "print(\"✅ Generated synthetic dense vectors for text (dim=768) and image (dim=512).\")\n",
    "print(f\"Text vector sample: {sample_text_vector[:5]}...\")\n",
    "print(f\"Image vector sample: {sample_image_vector[:5]}...\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 🎉 All database and vector verification tests completed successfully!"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}